# UFC Fight Prediction and Betting Market Evaluation

## Core Research Question

Can historical UFC fight data be used to build a probabilistic model for fight outcomes, and can those probabilities identify betting opportunities that the market has mispriced?

I approached this project as more than a winner-prediction problem. Predicting which fighter is more likely to win is useful, but the more interesting question is whether a model can produce probabilities that are good enough to compare against sportsbook odds. In other words, can the model find situations where its estimated probability differs meaningfully from the market’s implied probability?

## Data Sources

Methodology update after the final audit: the predictive workflow now restricts modeling, odds matching, betting analysis, and walk-forward research to fights on or after `2010-01-01`. The full historical table is still retained for EDA and reference, but pre-2010 fights are excluded from predictive modeling because the older source appears to break the red/blue corner convention.

I used two main UFC datasets.

The fight-level data came from:  
https://raw.githubusercontent.com/komaksym/UFC-DataLab/refs/heads/main/data/stats/stats_raw.csv

This dataset contains UFC fight data up to March 14, 2026. It includes fighter names, dates, results, method of victory, round and time, referee, division, event name, location, and fight statistics such as strikes, takedowns, submission attempts, reversals, and control time.

The fighter-level data came from:  
https://github.com/komaksym/UFC-DataLab/blob/main/data/external_data/raw_fighter_details.csv

This dataset contains fighter attributes such as name, height, weight, reach, stance, and career-level statistics like striking rate, takown defense, takedown average, and submission average.

## Preventing Data Leakage

The first major issue I ran into was the information barrier. The fighter statistics dataset contains career-level statistics for each fighter. For example, if I am predicting a fight from 2022, the dataset today may include fights that happened after 2022. At the time of that fight, those future fights would not have been known.

Using those career statistics directly would create data leakage because the model would be using future information to predict past outcomes. This would make the model look more accurate than it would be in a real prediction setting.

To avoid this, I structured the dataset so that each fight only used information available before that fight occurred. I sorted fights chronologically and calculated cumulative pre-fight statistics for each fighter using only their previous UFC fights. For example, a fighter’s pre-fight wins, losses, strikes, takedowns, knockdowns, and control time were all based only on fights before the current fight.

I treated height, reach, and stance as mostly static attributes. This is an assumption and a limitation. Fighters can change weight classes, and some fighters switch stances or evolve stylistically over time. However, given the available data, I treated these as stable enough to include.

The first step, therefore, was not modeling. It was cleaning and restructuring the data so that the model would only see what would have been known before each fight.

## Initial Feature Construction

After creating the leakage-safe dataset, I built features designed to compare the two fighters in each matchup. Many of the most useful features were difference features, such as:

- difference in age
- difference in total wins
- difference in total losses
- difference in win streak
- difference in reach
- difference in takedowns landed per fight
- difference in striking accuracy

I also created rate and efficiency features instead of relying only on raw totals. For example, strikes per minute are more informative than total strikes because fighters have different numbers of fights and different total cage time.

The goal was to construct features that reflected actual pre-fight advantages rather than simply giving the model raw fight records.

## Hypothesis 1: Experience and Record Matter

My first hypothesis was that experience and record should matter. Fighters with more wins, more UFC fights, or stronger win percentages may have an advantage.

To test this, I created features such as:

- total wins
- total losses
- total fights
- win percentage
- years in UFC
- fights per year
- win streak and loss streak

One important realization was that win percentage alone can be misleading. A 1–0 fighter has a 100% win rate, but that does not necessarily mean they are stronger than a 20–5 fighter. Total wins and total losses preserve sample-size information that win percentage loses.

Experience-related features ended up being among the strongest signals, especially age, total wins, total losses, and win streak. This suggested that both proven success and career wear matter.

## Hypothesis 2: Power and Durability Matter

Another hypothesis was that striking power should help predict fight outcomes. The difficult part was figuring out how to quantify power.

One possible approach would be to find punching-machine scores or some direct measure of force. However, this is not realistic. Most UFC fighters do not have publicly available punching-machine scores, and even if they did, hitting a machine outside the cage is very different from landing clean shots in a fight.

Instead, I tested several fight-based proxies:

- knockdowns per strike landed
- knockdowns per minute
- knockout wins per fight
- strike-type-adjusted knockdown rates

Each approach had tradeoffs. Knockdowns per strike rewards fighters who do more damage with fewer landed strikes, but it depends heavily on opponent durability. Knockout rate is intuitive, but it can penalize fighters who are powerful but usually win through grappling or decisions.

I also considered strike location. Head strikes are more directly connected to knockdowns than body or leg strikes, so treating every strike equally would oversimplify the problem. I created weighted strike-efficiency features and combined several normalized components into a power index.

However, this feature was still experimental. A power index is not a directly observed statistic; it is something I constructed based on assumptions about what should matter. Because of that, it could introduce bias if the formula does not capture true fight dynamics.

In validation, the power-related features did not improve the model much and sometimes made performance worse. This was surprising at first, but it likely reflects the high variance of knockout statistics and the fact that knockdowns depend heavily on opponent durability, fight style, and small-sample randomness.

## Hypothesis 3: Opponent Strength Matters

Raw records do not account for quality of competition. A fighter who is 5–1 against weaker opponents may not be better than a fighter who is 3–3 against elite opponents.

To address this, I tested opponent-strength features. The first versions of these features did not improve validation performance, so I built an Elo rating system.

Elo is commonly used in chess to estimate player skill over time. Each fighter has a rating that updates after every fight. Beating a highly rated opponent increases a fighter’s rating more than beating a weak opponent, while losing to a lower-rated opponent causes a larger decrease.

This was useful conceptually because Elo is time-aware and can be computed without future information. Each fighter’s pre-fight Elo only depended on fights that happened before the current fight.

The main limitation was handling UFC newcomers. A fighter’s first UFC fight is not necessarily their first professional fight. Many fighters enter the UFC after competing in Bellator, ONE, or regional promotions. Assigning every newcomer the same starting Elo can underestimate experienced debutants and overestimate true rookies.

I treated the Elo K-factor as a hyperparameter because it controls how aggressively ratings update after each fight. A larger K makes ratings move quickly, while a smaller K makes them smoother. I tested K values on the validation set and selected the one that minimized validation log loss.

Elo added some incremental signal compared with a clean baseline, but it did not beat the best validation model. This suggested that opponent-adjusted ratings are conceptually useful, but the UFC-only Elo implementation is limited. A better version would initialize ratings using pre-UFC records, prior promotion strength, or even market expectations.

## Hypothesis 4: Recent Form Matters

Another hypothesis was that fighter skill is not static. Fighters improve, decline, change camps, suffer injuries, and adapt their styles. Older fights may not represent current ability as well as recent fights.

To test this, I created two types of recency features:

- last-N fight statistics
- exponentially weighted historical statistics

The last-N features used only a fighter’s most recent fights. The exponential decay features gave more weight to recent fights and gradually reduced the influence of older fights.

I selected the last-N window and decay strength using validation performance.

To my surprise, recency features did not improve the model much. One likely reason is that the baseline features already captured some recent information through win streaks and efficiency metrics. Another reason is that many UFC fighters have a relatively small number of fights, so recent statistics can end up looking very similar to career statistics.

This suggested that recency is probably important in specific cases, such as layoffs, aging, or rapid improvement, but a simple universal recency weighting does not automatically improve performance.

## Hypothesis 5: Style, Matchups, and Cardio Matter

After testing broad feature groups, I shifted toward more specific MMA-based hypotheses. Instead of just adding more columns, I asked which fight dynamics should matter and tried to construct features to test them.

I tested several groups:

- striker versus grappler style scores
- finishing ability versus opponent durability
- short turnarounds and long layoffs
- recent form compared with career averages
- cardio and long-fight experience
- takedown offense versus opponent takedown defense

Each group was evaluated on the validation set using log loss and ROC AUC. The test set was not used.

The clearest improvement came from cardio-related features. I then narrowed the cardio block to identify which features were actually useful. The strongest cardio features were:

- difference in average fight duration
- difference in decision rate
- red fighter late-finish rate
- difference in five-round experience

A smaller two-feature version performed nearly as well, but the four-feature subset produced the best validation performance.

This reinforced one of the main lessons of the project: more features do not automatically improve a model. A small number of well-designed features can be more useful than a large group of noisy ones.

## Logistic Regression Baseline

I started with logistic regression because it is interpretable. The goal was not just to get predictions, but to understand which features were contributing signal.

I split the dataset chronologically into:

- training set: used to fit models
- validation set: used for feature selection and model comparison
- test set: held out until the final evaluation

A chronological split was important because the model is supposed to use past fights to predict future fights.

To evaluate features, I used:

- single-feature models
- leave-one-feature-out analysis
- leave-one-group-out analysis
- forward selection
- elastic net regularization

The best validation model used a forward-selected feature set with elastic net regularization. It achieved approximately:

- log loss: 0.663
- ROC AUC: 0.643
- accuracy: about 60%

This was a modest improvement over earlier baselines. It did not suggest a massive predictive edge, but it did show that the engineered features contained real signal.

The strongest features included age difference, total wins difference, win streak difference, and takedowns landed per fight difference. This suggested that youth, proven success, recent momentum, and grappling output were useful predictors.

## Final Logistic Model

After feature engineering and validation testing, I selected a final logistic regression model and evaluated it once on the hold-out test set.

The final logistic model achieved:

- ROC AUC: 0.649
- log loss: 0.677
- accuracy: 57.9%

Performance declined slightly from validation, which was expected because the test period had a different distribution. In particular, the red-corner win rate dropped from about 64.6% in training/validation to about 54.5% in the test set. This suggests that the model learned useful patterns, but some historical effects, especially red-corner advantage, were less stable in more recent fights.

## Hypothesis 6: Nonlinear Models Can Capture Interactions Better

After building the logistic model, I tested whether nonlinear models could improve performance. Logistic regression assumes a linear relationship between features and the log-odds of winning, but MMA outcomes may involve nonlinear effects and interactions.

For example, age may only become a serious disadvantage after a certain point, or reach advantage may matter differently depending on weight class or fighting style.

I tested two tree-based models:

- Random Forest
- XGBoost

To keep the comparison fair, both models used the same final feature set as the logistic model. Hyperparameters were selected using validation data only.

Random Forest improved over its initial baseline after tuning, but it still did not outperform logistic regression on validation. XGBoost also improved over its baseline, but did not beat the logistic benchmark either. This suggested that the engineered feature set already captured much of the usable signal in a form that logistic regression could model effectively.

However, the final hold-out comparison showed an important tradeoff. Logistic regression achieved the best post-cutoff log loss, ROC AUC, and Brier score, meaning it was the strongest final probability model on the consumed hold-out benchmark. Random Forest achieved the best post-cutoff accuracy, while its earlier betting pass was retained only as a historical exploratory comparison.

Because the next stage was a betting backtest, probability quality mattered more than ranking alone. For that reason, the main post-cutoff betting layer now uses Logistic Regression probabilities, while the older Random Forest backtest remains in the repo only for comparison.

## Adding Betting Odds

The original datasets did not include betting odds, so I added an external odds dataset:  
https://www.kaggle.com/datasets/jerzyszocik/ufc-betting-odds-daily-dataset

This dataset contains historical UFC betting odds dating back to 2010 and includes odds from multiple bookmakers.

Because there were multiple sources, I selected one consistent source instead of averaging odds across books. Averaging would create prices that are not directly tradable and could distort expected value calculations. The selected source was “zewnetrzne,” which means “external” in Polish and appears to represent an aggregated external odds feed.

I cleaned the odds data by standardizing fighter names and event dates, removing duplicates, converting decimal odds into implied probabilities, and normalizing probabilities to remove the sportsbook margin. This produced no-vig market probabilities.

The cleaned odds data was then merged with the modeling dataset by event date and fighter names, while accounting for red/blue corner ordering.

## Hypothesis 7: Predictive Signal Can Become Betting Edge

The final hypothesis was that if the model produced better probabilities than the market, it should be able to identify positive expected value bets.

This is a much harder task than predicting winners. A model can be directionally useful and still fail to beat market odds if the sportsbook already prices in the same information.

For each fight, I compared the Random Forest model probability to the no-vig market-implied probability. The difference represented the model’s estimated edge.

The betting strategy could bet on at most one side per fight:

- bet red if the red-side edge was highest and above the threshold
- bet blue if the blue-side edge was highest and above the threshold
- skip the fight otherwise

I tested multiple edge thresholds to see whether more selective betting improved performance.

I evaluated two staking approaches:

- flat betting, where each bet used the same fixed stake
- fractional Kelly betting, where bet size scaled with estimated edge

The model and features were frozen before this stage. Betting results were not used to retrain or tune the model.

## Betting Results

The betting backtest was negative across all tested thresholds and bet sizing strategies. Flat betting lost money at every threshold, and Kelly sizing performed worse because it amplified errors in the model probabilities.

This was one of the most important results of the project.

The main assumption that failed was that a model capable of predicting fight outcomes reasonably well would also identify profitable betting opportunities. In practice, betting markets already incorporate a large amount of information. Many of my strongest features, such as age, record, streaks, reach, and efficiency metrics, are likely already reflected in the odds.

The diagnostic analysis showed that larger model edges did not reliably produce better returns. Losses were concentrated especially in underdog and blue-side bets. This suggests that the model was not identifying true mispricings, even when it disagreed with the market.

The Kelly results also showed why calibration matters. Kelly betting assumes the model’s probabilities are accurate. If the probabilities are even slightly wrong, the strategy can overbet and lose money quickly.

## Limitations and Future Improvements

One major limitation is that the model relies mostly on aggregate statistics. These features do not fully capture stylistic interactions between fighters. In practice, fight outcomes often depend on how one fighter’s tendencies exploit another’s weaknesses. For example, a fighter who drops their hands may be especially vulnerable to an opponent with strong counter-striking, even if their overall statistics look strong.

A better model could incorporate more granular behavior-level data. Computer vision techniques could be used to analyze fight footage and extract information about movement, positioning, defensive habits, and strike patterns over time.

Another extension would be to use unstructured text data, such as analyst commentary, media coverage, or public sentiment. These sources may contain information about injuries, training camps, or matchup-specific concerns that are not captured in structured statistics.

Another limitation is that engineered features such as Elo ratings and power indices depend on modeling choices. Elo requires choosing an initial rating and update sensitivity, while power indices require choosing which statistics to include and how to weight them. These features may be useful, but they are not directly observed measurements.

The betting analysis also has limitations. The odds source was an aggregated external feed rather than a single sharp sportsbook, and the model did not account for line shopping across sportsbooks. A more realistic betting framework would compare available prices across books and evaluate whether the model could find value at the best available odds.

Finally, the odds data only covers fights from 2010 onward, reducing the sample size for betting analysis. UFC outcomes are also inherently noisy because of injuries, weight cuts, training conditions, stylistic matchups, and random fight events.

## Final Takeaway

The final result of this project is not a profitable betting system. Instead, the main finding is that building a reasonable predictive model is not enough to beat a betting market.

The model captured real signal in fight outcomes, but much of that signal appears to already be priced into the market. The betting backtest showed that the model did not identify consistent mispricings after accounting for sportsbook odds and probability calibration.

This project shows the difference between prediction and decision-making. A model can perform better than random chance and still fail to generate positive expected value. In market-based settings, the hard part is not only forecasting outcomes, but finding information that the market has not already priced in.
